# 00.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/00-big-picture/03-three-kinds-of-learning/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd
cust = pd.read_csv(data_path("shopwave_customers_small.csv"))

## Exercise 1

In [2]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
features = ["visits_last_month", "items_last_month", "days_since_last"]
X_scaled = StandardScaler().fit_transform(cust[features])

for k in [2, 3, 5]:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_scaled)
    print(f"\n--- k = {k} ---")
    print(cust.assign(segment=km.labels_).groupby("segment")[features].mean().round(1)
              .assign(n=pd.Series(km.labels_).value_counts().sort_index()))


--- k = 2 ---
         visits_last_month  items_last_month  days_since_last   n
segment                                                          
0                      2.2               2.0             31.2   5
1                     10.1               6.3             30.0  15

--- k = 3 ---
         visits_last_month  items_last_month  days_since_last   n
segment                                                          
0                      5.2               4.0             50.8   5
1                     10.8               6.6             25.4  12
2                      2.3               2.0             15.7   3

--- k = 5 ---
         visits_last_month  items_last_month  days_since_last  n
segment                                                         
0                      9.2               5.4             19.4  5
1                      2.0               2.0             54.5  2
2                      2.3               2.0             15.7  3
3                      8.2          

With `k=2` you get roughly "light users" (about 2 visits) vs "heavy users" (about 10), and recency barely differs between them. With `k=5` some groups have only two or three customers and differ by tiny amounts. The algorithm always obliges; whether the groups mean anything is your call. Module 06 gives you tools (elbow, silhouette) to choose *k* less arbitrarily.

## Exercise 2

In [3]:
from sklearn.tree import DecisionTreeClassifier, export_text
X_cheat = cust[features + ["churned"]]
y = cust["churned"]
cheat = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_cheat, y)
print(f"accuracy: {(cheat.predict(X_cheat) == y).mean():.0%}")
print(export_text(cheat, feature_names=list(X_cheat.columns)))

accuracy: 100%
|--- churned <= 0.50
|   |--- class: 0
|--- churned >  0.50
|   |--- class: 1



100%, with a single split on `churned` itself. The model "predicts" the answer by reading it. This is the extreme form of **data leakage**: a feature that contains the target. Real leaks are subtler (a column computed *after* the outcome), and Module 04 teaches you to hunt them.

## Exercise 3

Example answers:

- *"How many items will this customer buy next month?"* Supervised, regression. Label = count of items in the following calendar month, built from the orders table.
- *"Which products are usually bought in the same basket?"* Unsupervised, association rules.
- *"What order of products on the homepage maximises clicks over a month?"* Reinforcement (or an A/B test, which is the supervised cousin).